# agentkit provider check

Run this notebook after changing `.env` to see which provider and model the projects will use, and to check that it works. It:

1. reports the automatic provider selection, without ever showing secrets;
2. runs the offline test-suite, which covers the OpenAI, Gemini, Ollama and Hugging Face adapters with no keys or network needed;
3. makes **live** calls to the selected model: one structured output and one tool-calling task; and
4. prints the provenance report (live vs cached calls, tokens, cost).

In [1]:
import os
os.environ["TQDM_DISABLE"] = "1"
import subprocess, sys, json, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent))
from pydantic import BaseModel
from agentkit import LLM, Settings, configure_logging, LOCAL_HF_MODELS, DEFAULT_API_MODELS
from agentkit.config import free_memory_gb, discover_ollama
log = configure_logging("outputs")

## 1. Configuration and automatic selection

In [2]:
s = Settings.from_env()
print(json.dumps(s.public(), indent=1))
free, device = free_memory_gb()
print(f"\nFree {device} memory: {free:.1f} GB | Ollama server at {s.ollama_host}: {discover_ollama(s.ollama_host) or 'not reachable'}")
print("Local model tiers (best first):", [m for m, _ in LOCAL_HF_MODELS], "| default API models:", DEFAULT_API_MODELS)

{
 "provider": "auto",
 "model": null,
 "openai_api_key": "not set",
 "openai_model": "gpt-4.1-mini",
 "openai_base_url": null,
 "gemini_api_key": "not set",
 "gemini_model": "gemini-3.5-flash-lite",
 "ollama_host": "http://localhost:11434",
 "ollama_model": null,
 "hf_model": null,
 "cache_mode": "off",
 "fallback_to_local": true,
 "max_retries": 4,
 "min_interval": 0.0,
 "fallback_models": null,
 "timeout": 120.0,
 "seed": 0,
 "price_input_per_m": null,
 "price_output_per_m": null
}



Free cpu memory: 7.6 GB | Ollama server at http://localhost:11434: not reachable
Local model tiers (best first): ['Qwen/Qwen3-8B', 'Qwen/Qwen3-4B', 'Qwen/Qwen3-1.7B', 'Qwen/Qwen3-0.6B'] | default API models: {'openai': 'gpt-4.1-mini', 'gemini': 'gemini-3.5-flash-lite'}


## 2. Offline test-suite (all four providers)

In [3]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-rA", "--color=no", "-p", "no:cacheprovider", "tests"], capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if l.startswith(("PASSED", "FAILED", "SKIPPED", "ERROR")) or "passed" in l))
assert r.returncode == 0, r.stdout[-2000:]

PASSED tests/test_agentkit.py::test_auto_selection_order
PASSED tests/test_agentkit.py::test_secrets_never_exposed
PASSED tests/test_agentkit.py::test_local_model_tiers_and_ollama_preference
PASSED tests/test_agentkit.py::test_invalid_settings_rejected
PASSED tests/test_agentkit.py::test_openai_adapter_chat_json_and_tools
PASSED tests/test_agentkit.py::test_openai_bad_key_falls_back_to_local
PASSED tests/test_agentkit.py::test_reasoning_models_omit_temperature
PASSED tests/test_agentkit.py::test_ollama_discovery_and_adapter
PASSED tests/test_agentkit.py::test_gemini_adapter
PASSED tests/test_agentkit.py::test_cache_off_by_default_and_replays_are_labelled
PASSED tests/test_agentkit.py::test_transient_errors_retry_then_succeed
PASSED tests/test_agentkit.py::test_same_provider_model_fallback_before_local
PASSED tests/test_agentkit.py::test_retry_delay_honours_provider_hint
PASSED tests/test_agentkit.py::test_structured_output_repair_and_failure
PASSED tests/test_agentkit.py::test_agent_lo

## 3. Live calls to the selected model

In [4]:
llm = LLM()
print(llm.describe())

class Request(BaseModel):
    tickers: list
    year: int

req = llm.structured([{"role": "system", "content": 'Extract the request as JSON: {"tickers": [stock tickers], "year": int}'},
                      {"role": "user", "content": "How did Microsoft and Apple compare in 2017?"}], Request)
print("\nstructured output:", req)

PRICES = {("AAPL", "2017-03-01"): 139.79, ("MSFT", "2017-03-01"): 64.94}

def get_close(ticker: str, date: str) -> dict:
    """Closing price of a stock on a date.

    Args:
        ticker: stock ticker, e.g. AAPL
        date: date as YYYY-MM-DD
    """
    if (ticker, date) not in PRICES:
        raise KeyError(f"no price for {ticker} on {date}")
    return {"ticker": ticker, "date": date, "close": PRICES[(ticker, date)]}

run = llm.run_agent([{"role": "user", "content": "Which closed higher on 2017-03-01, Apple (AAPL) or Microsoft (MSFT)? Use the tool."}], [get_close])
for st in run.steps:
    print(f"tool call: {st.tool}({st.arguments}) -> {st.result}")
print("final answer:", run.final_text)

Provider: gemini   Model: gemini-3.5-flash-lite
Why: GOOGLE_API_KEY/GEMINI_API_KEY is set
API keys: OpenAI not set, Gemini set | cache: off (every call is live) | fallback to local: on
.env files loaded: ['Agentic AI/.env']


/usr/local/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



structured output: tickers=['MSFT', 'AAPL'] year=2017


/usr/local/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


tool call: get_close({'ticker': 'AAPL', 'date': '2017-03-01'}) -> {"ticker": "AAPL", "date": "2017-03-01", "close": 139.79}
tool call: get_close({'ticker': 'MSFT', 'date': '2017-03-01'}) -> {"ticker": "MSFT", "date": "2017-03-01", "close": 64.94}
final answer: On 2017-03-01, Apple (AAPL) closed higher at $139.79 compared to Microsoft (MSFT) which closed at $64.94.


## 4. Provenance

In [5]:
llm.report();

LLM provenance for this run
  gemini/gemini-3.5-flash-lite: GOOGLE_API_KEY/GEMINI_API_KEY is set
  LIVE  gemini/gemini-3.5-flash-lite: 3 calls, 403 in / 140 out tokens, 9s
  live calls: 3 | replayed from cache: 0 | retries: 0 | fallbacks: 0
